In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image


from pc import PS
from modules import ADC,DAC,CHIP
from cimCommand import CMD,CmdData,Packet
from cimCommand.singleCmdInfo import *

from util import plot_v_cond

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=3),init=True)
chip.set_device_cfg(deviceType=0)

In [ ]:
delay = 100
delay2 = 100
pkts=Packet()
pkts.append_cmdlist([
    CMD(OP_BANK_CFG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_CIM_DATA_CFG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(REG_CLK_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(LATCH_CLK_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_ADC_AVRG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_DAC_AVRG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_ROW_PULSE_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_COL_PULSE_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_CIM_RSTN_DONE_DELAY_CYC,command_data=CmdData(delay)),

    CMD(CIM_RSTN_CYC,command_data=CmdData(delay2)),
    CMD(LATCH_CLK_CYC,command_data=CmdData(delay2)),
    CMD(REG_CLK_CYC,command_data=CmdData(delay2)),
    CMD(LATCH_CYC,command_data=CmdData(delay2)),
],mode=1)
chip.ps.send_packets(pkts)

In [ ]:
# ins_data=[CMD(PL_DAC_V,command_data=CmdData((i+DAC_INFO.INDEX_START)<<16)) for i in range(1)]
# chip.execute_ins(ins_data=ins_data,ins_ram_start=0)
need_read = np.zeros((256,256),dtype=bool)
need_read[120,:128] = 1
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)

pkts=Packet()
pkts.append_cmdlist([
    CMD(INS_RUN_TIME_L32),
],mode=2)
chip.ps.send_packets(pkts,recv=False)

message = chip.ps.receive_packet(4)

tmp = message.hex()[6:8]+message.hex()[4:6]+message.hex()[2:4] + message.hex()[0:2]
print(tmp)
data = int(tmp, 16)
print(data*1e-8)

In [ ]:
pkts=Packet()
pkts.append_single([
    CMD(PL_RAM_ADDR,command_data=CmdData(128)),
    CMD(PL_DATA_LENGTH,command_data=CmdData(1))
],mode=6)
chip.ps.send_packets(pkts,recv=False)

tia16_length = 64
tia_num = 16
tia_length = 4
# 接收信息, num条dout_ram值, 每条dout_ram长为256/8=32B
message = chip.ps.receive_packet(32)

In [ ]:
delay = 10
delay2 = 100
pkts=Packet()
pkts.append_cmdlist([
    CMD(OP_BANK_CFG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_CIM_DATA_CFG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(REG_CLK_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(LATCH_CLK_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_ADC_AVRG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_DAC_AVRG_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_ROW_PULSE_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_COL_PULSE_DONE_DELAY_CYC,command_data=CmdData(delay)),
    CMD(OP_CIM_RSTN_DONE_DELAY_CYC,command_data=CmdData(delay)),

    CMD(CIM_RSTN_CYC,command_data=CmdData(delay2)),
    CMD(LATCH_CLK_CYC,command_data=CmdData(delay2)),
    CMD(REG_CLK_CYC,command_data=CmdData(delay2)),
    CMD(LATCH_CYC,command_data=CmdData(delay2)),
],mode=1)
chip.ps.send_packets(pkts)

In [ ]:
need_read = np.zeros((256,256),dtype=bool)
need_read[120,:] = 1
data = chip.get_data(crossbar=need_read,from_row=True)

In [ ]:
need_read = np.zeros((256,256),dtype=bool)
need_read[120,:128] = 1
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

In [ ]:
message = chip.ps.receive_packet(4)

In [ ]:
ins_data=[CMD(PL_DAC_V,command_data=CmdData((i+DAC_INFO.INDEX_START)<<16)) for i in range(1)]
chip.execute_ins(ins_data=ins_data,ins_ram_start=0)

In [ ]:
pkts=Packet()
pkts.append_cmdlist([
    CMD(INS_RUN_TIME_L32),
    # CMD(INS_RUN_TIME_H32),
],mode=2)
chip.ps.send_packets(pkts,recv=False)

message = chip.ps.receive_packet(4)

tmp = message.hex()[6:8]+message.hex()[4:6]+message.hex()[2:4] + message.hex()[0:2]
print(tmp)
data = int(tmp, 16)
print(data*1e-8)

message = chip.ps.receive_packet(4)
print(message.hex())
data = int(message.hex(), 16)
print(data)

In [ ]:
import time

pkts=Packet()
pkts.append_cmdlist([
    CMD(INS_RUN_TIME_L32),
],mode=3)

start_time = time.perf_counter()
chip.ps.send_packets(pkts,recv=False)
message = chip.ps.receive_packet(3)
end_time = time.perf_counter()
elapsed_time = end_time - start_time  
print(f"用时: {elapsed_time:.6f} seconds")                         

In [ ]:
pkts=Packet()
pkts.append_cmdlist([
    CMD(INS_RUN_TIME_L32),
    CMD(INS_RUN_TIME_H32),
],mode=2)
chip.ps.send_packets(pkts,recv=False)

message = chip.ps.receive_packet(4)

tmp = message.hex()[6:8]+message.hex()[4:6]+message.hex()[2:4] + message.hex()[0:2]
print(tmp)
data = int(tmp, 16)
print(data*1e-8)

message = chip.ps.receive_packet(4)
print(message.hex())
data = int(message.hex(), 16)
print(data)

In [ ]:
result = []
for i in range(40):
    voltage_base = chip.read_point2(data=data, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(data=data, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    result.append(cond_sub_base[120,:])

# result = []
# for i in range(40):
#     voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
#     voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
#     cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
#     result.append(cond_sub_base[120,:])

In [ ]:
average = np.zeros((1,256))
for k,v in enumerate(result):
    average += v
average /=40

sum = 0
for k,v in enumerate(result):
    tmp = np.sum(abs(v-average)>50)
    print(tmp)
    sum +=tmp

print(sum)
plt.figure(figsize=(12,8))
for k,v in enumerate(result):
    plt.plot(v,label=f"{k}")

plt.plot(average[0,:],label = "average")
plt.legend()
plt.show()